# 🎬 Estúdio — rodar no Kaggle (2x T4, o mais rápido de graça)

Antes de rodar, no painel da direita (**Settings**):
1. **Accelerator → GPU T4 x2**.
2. **Internet → On** (precisa ter o telefone verificado na conta do Kaggle).
3. (Opcional, para o Space) **Add-ons → Secrets**: crie `HF_TOKEN`.

O Kaggle dá ~30 h de GPU por semana, com sessões de até 12 h. Com 2 GPUs, o codificador de texto
fica numa e o modelo de vídeo na outra, então tudo roda em fp16 sem quantização.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Baixa o código e instala as dependências
RAMO = "estudio-video"
!rm -rf Jarvis && git clone -q -b {RAMO} --depth 1 https://github.com/gutodrac-spec/Jarvis.git
%cd Jarvis/estudio
!pip install -q -r requirements.txt

In [ ]:
# Pré-carrega o modelo (~16 GB na 1ª vez) e sobe a interface + API com link público.
# Quando aparecer "Running on public URL: https://….gradio.live", abra o link
# ou cole ele na página web/index.html.
!ESTUDIO_SHARE=1 ESTUDIO_PRECARREGAR=1 python app.py

## Publicar como Hugging Face Space (opcional)
Precisa do segredo `HF_TOKEN` (token *write*). O Space grátis só tem CPU, que é lento demais para vídeo:
escolha uma GPU nas configurações do Space ou use **ZeroGPU** (conta PRO). Pare a célula que roda o app antes.

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi

NOME_DO_SPACE = "estudio"  # vira https://huggingface.co/spaces/<seu-usuario>/estudio

api = HfApi(token=HF_TOKEN)
repo = f"{api.whoami()['name']}/{NOME_DO_SPACE}"
api.create_repo(repo, repo_type="space", space_sdk="gradio", exist_ok=True)
with open("/tmp/README.md", "w") as f:
    f.write("---\ntitle: Estúdio\nemoji: 🎬\nsdk: gradio\napp_file: app.py\npinned: false\n---\n\n"
            "Texto ou imagem para vídeo com presets de câmera.\n")
api.upload_folder(repo_id=repo, repo_type="space", folder_path=".",
                  allow_patterns=["app.py", "presets.py", "requirements.txt"])
api.upload_file(repo_id=repo, repo_type="space", path_or_fileobj="/tmp/README.md", path_in_repo="README.md")
print("Space:", f"https://huggingface.co/spaces/{repo}")